In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.datasets import make_moons
from mlxtend.plotting import plot_decision_regions

#load the dataset
df = sns.load_dataset('penguins')
columns = df.columns

## Explore the dataset

In [ ]:
# how many rows
print(f"rows: {len(df.index)}\n")

# how many null values each column has
for column in columns:
	print(f"{column}: {df[column].isna().sum()} null values")
print("\n")

# average penguin body mass
print(f"average penguin body mass: {df['body_mass_g'].mean()}\n")

# number of species
print(f"number of penguin species: {df['species'].unique().size}\n")

# min amd max flipper length
print(f"min length: {df['flipper_length_mm'].min()}, max length: {df['flipper_length_mm'].max()}")


## Data Cleaning & Preprocessing

In [ ]:
# remove data points with missing sex values
cleanDf = df.copy(deep = True)
cleanDf.dropna(subset=["sex"])

# now impute the missing columns
imputedColumnNames = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
imputedColumns = cleanDf[imputedColumnNames]

imputer = SimpleImputer(strategy="median")
imputedData = imputer.fit_transform(imputedColumns)
dfImputedData = pd.DataFrame(imputedData, columns=imputedColumnNames, index=cleanDf.index)

cleanDf[imputedColumnNames] = dfImputedData

# scale
numericColumnNames = cleanDf.select_dtypes(include='number').columns
scaler = StandardScaler()
scaledData = scaler.fit_transform(cleanDf[numericColumnNames])
dfScaledData = pd.DataFrame(scaledData, columns=numericColumnNames)
cleanDf[numericColumnNames] = dfScaledData

# one hot encoding
cleanDf = pd.get_dummies(cleanDf, columns=["island", "sex"], drop_first=True)
cleanDf.head()


## Train/Test Split

In [ ]:
X = cleanDf.drop("species", axis=1)
y = cleanDf["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1337)

## KNN

In [ ]:
knnColumnNames = ["bill_length_mm", "bill_depth_mm"]

def trainKnn(neighbors, columnNames):

	#convert string labels to ints so that the decision boundary could be plotted
	trainLabels = LabelEncoder().fit_transform(y_train)
	testLabels =  LabelEncoder().fit_transform(y_test)

	knn = KNeighborsClassifier(n_neighbors = neighbors)
	knn.fit(X_train[columnNames].to_numpy(), trainLabels)
	knn.predict(X_test[columnNames].to_numpy())
	print(f"KNN accuracy for {neighbors} neighbors on {columnNames}: {accuracy_score(knn.predict(X_test[knnColumnNames]), testLabels)}")

	return knn
	
def plotDecisionBoundary(model, columnNames):
	plot_decision_regions(X[columnNames].to_numpy(), LabelEncoder().fit_transform(y), clf=model, legend = 2)
	plt.show();

#def plotDecisionBoundary2(model, columnNames):
	#convert string labels to ints so that the decision boundary could be plotted
	#trainLabels = LabelEncoder().fit_transform(y_train)
	#testLabels =  LabelEncoder().fit_transform(y_test)
	#trainData = X_train[columnNames].to_numpy()
	#testData = X_test[columnNames].to_numpy()

	#model.fit(trainData, trainLabels)
	#preds = model.predict(testData)
	#accuracy = accuracy_score(preds, testLabels)
	#print(f"Accuracy: {accuracy}")

	#plot_decision_regions(trainData, LabelEncoder().fit_transform(y_train), clf=model, legend = 0)
	#plt.show();

knn5 = trainKnn(5, knnColumnNames)
knn10 = trainKnn(10, knnColumnNames)
knn20 = trainKnn(20, knnColumnNames)

#plotting is slow
plotDecisionBoundary(knn5, knnColumnNames)
#plotKnn(knn20, knnColumnNames)

## Perceptron

In [ ]:
perceptron = Perceptron()
perceptron.fit(X_train, y_train)
perceptron_preds = perceptron.predict(X_test)
print(f"Accuracy for perceptron: {accuracy_score(y_test, perceptron_preds)}")

# plot the decision boundary for beaks, don't forget to transform all labels
perceptron.fit(X_train[knnColumnNames], LabelEncoder().fit_transform(y_train))
perceptron_preds = perceptron.predict(X_test[knnColumnNames])
print(f"Accuracy for perceptron on {knnColumnNames}: {accuracy_score(LabelEncoder().fit_transform(y_test), perceptron_preds)}")
plotDecisionBoundary(perceptron, knnColumnNames)


## Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)
print(f"Accuracy for random forest: {accuracy_score(y_test, preds)}")

## KFold

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1) Create dataset
X, y = datasets.make_regression(
    n_samples=100,
    n_features=5,
    n_informative=5,
    noise=10,
    random_state=0
)

# 2) K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = LinearRegression()
fold_scores = []

for train_idx, test_idx in kf.split(X):
    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    score = r2_score(y_test, preds)
    fold_scores.append(score)

print("K-Fold R² scores:", [round(s, 4) for s in fold_scores])
print("Mean R²:", round(np.mean(fold_scores), 4))

# 3) Single Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

single_model = LinearRegression()
single_model.fit(X_train, y_train)

y_pred = single_model.predict(X_test)
single_r2 = r2_score(y_test, y_pred)

print("\nSingle split R²:", round(single_r2, 4))

# 4) Comparison
print("\n=== Comparison ===")
print("Mean K-Fold R² :", round(np.mean(fold_scores), 4))
print("Single Split R²:", round(single_r2, 4))